# TP2 - Synchronisation de bases distribuees avec RabbitMQ

Ce notebook reprend le code Java du TP2 en cellules, avec une explication **ligne par ligne** pour la validation.

## Objectif du TP
- Deux agences (`BO1`, `BO2`) envoient leurs ventes vers le siege (`HO`) via RabbitMQ.
- Le siege consomme les messages et insere les donnees dans sa base consolidee.

## Ordre des fichiers expliques
1. `AppConfig.java`
2. `SaleRecord.java`
3. `MessageCodec.java`
4. `DbUtil.java`
5. `BranchProducer.java`
6. `HeadOfficeConsumer.java`


## 1) AppConfig.java


In [ ]:
public class AppConfig {

    // RabbitMQ
    public static final String RABBIT_HOST = "localhost";
    public static final int RABBIT_PORT = 5672;
    public static final String RABBIT_USER = "guest";
    public static final String RABBIT_PASS = "guest";

    public static final String EXCHANGE_NAME = "sales.sync";
    public static final String EXCHANGE_TYPE = "direct";
    public static final String HO_QUEUE = "ho.sales.queue";

    public static final String RK_BO1 = "bo1.sales";
    public static final String RK_BO2 = "bo2.sales";

    // PostgreSQL
    public static final String DB_HOST = "localhost";
    public static final int DB_PORT = 5432;
    public static final String DB_USER = "postgres";
    public static final String DB_PASS = "postgres";

    public static String jdbcUrl(String dbName) {
        return "jdbc:postgresql://" + DB_HOST + ":" + DB_PORT + "/" + dbName;
    }
}


## Explication ligne par ligne - AppConfig.java
- **L1**: declare la classe `AppConfig` qui centralise les constantes de configuration.
- **L2**: ligne vide pour separer visuellement.
- **L3**: commentaire de section pour RabbitMQ.
- **L4**: host RabbitMQ (`localhost`).
- **L5**: port RabbitMQ (`5672`).
- **L6**: utilisateur RabbitMQ.
- **L7**: mot de passe RabbitMQ.
- **L8**: 
- **L9**: nom de l'exchange utilise pour la synchro des ventes.
- **L10**: type d'exchange (`direct`) pour router par cle exacte.
- **L11**: nom de la file du siege.
- **L12**: 
- **L13**: routing key pour la branche 1.
- **L14**: routing key pour la branche 2.
- **L15**: 
- **L16**: commentaire de section pour PostgreSQL.
- **L17**: host PostgreSQL.
- **L18**: port PostgreSQL (`5432`).
- **L19**: utilisateur PostgreSQL.
- **L20**: mot de passe PostgreSQL.
- **L21**: 
- **L22**: methode utilitaire qui construit l'URL JDBC selon le nom de base.
- **L23**: retourne une URL JDBC complete (`jdbc:postgresql://host:port/db`).
- **L24**: fin de la methode.
- **L25**: fin de la classe.


## 2) SaleRecord.java


In [ ]:
import java.math.BigDecimal;
import java.time.LocalDateTime;

public class SaleRecord {

    public String branchId;
    public int saleId;
    public String productName;
    public int quantity;
    public BigDecimal unitPrice;
    public LocalDateTime saleDate;

    public SaleRecord(String branchId, int saleId, String productName,
                      int quantity, BigDecimal unitPrice, LocalDateTime saleDate) {
        this.branchId = branchId;
        this.saleId = saleId;
        this.productName = productName;
        this.quantity = quantity;
        this.unitPrice = unitPrice;
        this.saleDate = saleDate;
    }

    @Override
    public String toString() {
        return branchId + " | " + saleId + " | " + productName;
    }
}


## Explication ligne par ligne - SaleRecord.java
- **L1**: importe `BigDecimal` pour les montants monetaires precis.
- **L2**: importe `LocalDateTime` pour la date/heure de vente.
- **L3**: 
- **L4**: declare la classe `SaleRecord` (modele d'une vente).
- **L5**: 
- **L6**: identifiant de branche (ex: `BO1`).
- **L7**: identifiant unique de vente.
- **L8**: nom du produit.
- **L9**: quantite vendue.
- **L10**: prix unitaire du produit.
- **L11**: date/heure de la vente.
- **L12**:
- **L13**: debut du constructeur avec tous les champs utiles.
- **L14**: suite de la signature du constructeur pour rester lisible.
- **L15**: affecte `branchId` recu au champ objet.
- **L16**: affecte `saleId`.
- **L17**: affecte `productName`.
- **L18**: affecte `quantity`.
- **L19**: affecte `unitPrice`.
- **L20**: affecte `saleDate`.
- **L21**: fin du constructeur.
- **L22**: 
- **L23**: annotation indiquant une redefinition de methode heritee.
- **L24**: debut de `toString()` pour affichage simplifie.
- **L25**: retourne une representation courte de la vente.
- **L26**: fin de `toString()`.
- **L27**: fin de la classe.


## 3) MessageCodec.java


In [ ]:
import java.math.BigDecimal;
import java.time.LocalDateTime;

public class MessageCodec {

    public static String encodeSaleRecord(SaleRecord saleRecord) {
        return s.branchId + ";" +
               s.saleId + ";" +
               s.productName + ";" +
               s.quantity + ";" +
               s.unitPrice + ";" +
               saleRecord.saleDate;
    }

    public static SaleRecord decodeSaleRecord(String payload) {
        String[] p = msg.split(";");

        return new SaleRecord(
                parts[0],
                Integer.parseInt(parts[1]),
                parts[2],
                Integer.parseInt(parts[3]),
                new BigDecimal(parts[4]),
                LocalDateTime.parse(parts[5])
        );
    }
}


## Explication ligne par ligne - MessageCodec.java
- **L1**: importe `BigDecimal` pour reconstruire le prix au decode.
- **L2**: importe `LocalDateTime` pour reconstruire la date.
- **L3**: 
- **L4**: declare la classe utilitaire de serialisation/deserialisation.
- **L5**: 
- **L6**: methode `encode` qui transforme un `SaleRecord` en texte.
- **L7**: commence la concatenation avec `branchId` + separateur `;`.
- **L8**: ajoute `saleId` + `;`.
- **L9**: ajoute `productName` + `;`.
- **L10**: ajoute `quantity` + `;`.
- **L11**: ajoute `unitPrice` + `;`.
- **L12**: ajoute `saleDate` en dernier champ.
- **L13**: fin de `encode`.
- **L14**: 
- **L15**: methode `decode` qui reconstruit un objet depuis le texte.
- **L16**: decoupe le message par `;` dans un tableau `p`.
- **L17**: 
- **L18**: cree un nouveau `SaleRecord` a partir du tableau.
- **L19**: `p[0]` devient `branchId`.
- **L20**: convertit `p[1]` en entier pour `saleId`.
- **L21**: `p[2]` devient `productName`.
- **L22**: convertit `p[3]` en entier pour `quantity`.
- **L23**: convertit `p[4]` en `BigDecimal` pour `unitPrice`.
- **L24**: convertit `p[5]` en `LocalDateTime` pour `saleDate`.
- **L25**: fin de l'appel constructeur.
- **L26**: fin de `decode`.
- **L27**: fin de la classe.


## 4) DbUtil.java


In [ ]:
import java.sql.Connection;
import java.sql.DriverManager;

public class DbUtil {

    static {
        try {
            Class.forName("org.postgresql.Driver");
        } catch (Exception e) {
            throw new RuntimeException("Driver PostgreSQL introuvable", e);
        }
    }

    public static Connection openConnection(String databaseName) throws Exception {
        return DriverManager.getConnection(
                AppConfig.jdbcUrl(databaseName),
                AppConfig.DB_USER,
                AppConfig.DB_PASS
        );
    }
}


## Explication ligne par ligne - DbUtil.java
- **L1**: importe `Connection` JDBC.
- **L2**: importe `DriverManager` pour ouvrir des connexions.
- **L3**: ligne vide.
- **L4**: declare la classe utilitaire DB.
- **L5**: ligne vide.
- **L6**: bloc statique execute une seule fois au chargement de classe.
- **L7**: debut du `try` pour charger le driver PostgreSQL.
- **L8**: force le chargement de `org.postgresql.Driver`.
- **L9**: capture toute exception de chargement.
- **L10**: relance une erreur claire si driver introuvable.
- **L11**: fin du `catch`.
- **L12**: fin du bloc statique.
- **L13**: ligne vide.
- **L14**: methode utilitaire pour obtenir une connexion vers `dbName`.
- **L15**: retourne une connexion ouverte via JDBC.
- **L16**: URL JDBC construite par `AppConfig.jdbcUrl(databaseName)`.
- **L17**: utilisateur DB depuis la config.
- **L18**: mot de passe DB depuis la config.
- **L19**: fin de l'appel `getConnection`.
- **L20**: fin de la methode.
- **L21**: fin de la classe.


## 5) BranchProducer.java


In [ ]:
import com.rabbitmq.client.Channel;
import com.rabbitmq.client.ConnectionFactory;
import com.rabbitmq.client.AMQP;

import java.nio.charset.StandardCharsets;
import java.sql.*;
import java.time.LocalDateTime;

public class BranchProducer {

    public static void main(String[] args) throws Exception {

        if (args.length != 3) {
            System.out.println("Usage: BO1 sales_bo1 bo1.sales");
            return;
        }

        String branchId = args[0];
        String dbName = args[1];
        String routingKey = args[2];

        ConnectionFactory factory = new ConnectionFactory();
        factory.setHost(AppConfig.RABBIT_HOST);

        com.rabbitmq.client.Connection mqConnection = factory.newConnection();
        Channel channel = mqConnection.createChannel();

        java.sql.Connection dbConnection = DbUtil.openConnection(dbName);

        channel.exchangeDeclare(AppConfig.EXCHANGE_NAME, "direct", true);
        channel.queueDeclare(AppConfig.HO_QUEUE, true, false, false, null);

        channel.queueBind(AppConfig.HO_QUEUE, AppConfig.EXCHANGE_NAME, AppConfig.RK_BO1);
        channel.queueBind(AppConfig.HO_QUEUE, AppConfig.EXCHANGE_NAME, AppConfig.RK_BO2);

        String select = "SELECT * FROM product_sales WHERE synced = false";
        String update = "UPDATE product_sales SET synced = true WHERE sale_id = ?";

        PreparedStatement psSelect = dbConnection.prepareStatement(select);
        PreparedStatement psUpdate = dbConnection.prepareStatement(update);

        ResultSet rs = psSelect.executeQuery();

        while (rs.next()) {

            SaleRecord s = new SaleRecord(
                    branchId,
                    rs.getInt("sale_id"),
                    rs.getString("product_name"),
                    rs.getInt("quantity"),
                    rs.getBigDecimal("unit_price"),
                    rs.getTimestamp("sale_date").toLocalDateTime()
            );

            String msg = MessageCodec.encodeSaleRecord(s);

            channel.basicPublish(
                    AppConfig.EXCHANGE_NAME,
                    routingKey,
                    new AMQP.BasicProperties.Builder().deliveryMode(2).build(),
                    msg.getBytes(StandardCharsets.UTF_8)
            );

            psUpdate.setInt(1, s.saleId);
            psUpdate.executeUpdate();

            System.out.println("Envoye: " + s);
        }

        rs.close();
        psSelect.close();
        psUpdate.close();
        dbConnection.close();
        channel.close();
        mqConnection.close();
    }
}


## Explication ligne par ligne - BranchProducer.java
- **L1**: importe l'interface `Channel` RabbitMQ.
- **L2**: importe `ConnectionFactory` pour creer une connexion RabbitMQ.
- **L3**: importe `AMQP` pour definir les proprietes du message.
- **L4**: ligne vide.
- **L5**: importe UTF-8 pour encoder le message texte en bytes.
- **L6**: importe les classes SQL (Connection, PreparedStatement, ResultSet...).
- **L7**: importe `LocalDateTime` (non utilise explicitement ici).
- **L8**: ligne vide.
- **L9**: declare la classe `BranchProducer` (producteur BO).
- **L10**: ligne vide.
- **L11**: point d'entree principal du programme.
- **L12**: ligne vide.
- **L13**: verifie qu'on a exactement 3 arguments en entree.
- **L14**: affiche l'usage attendu si arguments incorrects.
- **L15**: termine le programme en cas d'erreur d'usage.
- **L16**: fin du bloc `if`.
- **L17**: ligne vide.
- **L18**: lit l'ID branche depuis `args[0]`.
- **L19**: lit le nom de base BO depuis `args[1]`.
- **L20**: lit la routing key depuis `args[2]`.
- **L21**: ligne vide.
- **L22**: cree une factory RabbitMQ.
- **L23**: configure le host RabbitMQ.
- **L24**: ligne vide.
- **L25**: ouvre la connexion RabbitMQ.
- **L26**: ouvre un channel AMQP sur cette connexion.
- **L27**: ligne vide.
- **L28**: ouvre la connexion vers la base de la branche.
- **L29**: ligne vide.
- **L30**: declare l'exchange (durable).
- **L31**: declare la queue HO (durable, non exclusive, non auto-delete).
- **L32**: ligne vide.
- **L33**: bind la queue HO avec la routing key de BO1.
- **L34**: bind la queue HO avec la routing key de BO2.
- **L35**: ligne vide.
- **L36**: requete pour lire les ventes non synchronisees.
- **L37**: requete pour marquer une vente comme synchronisee.
- **L38**: ligne vide.
- **L39**: prepare la requete de selection.
- **L40**: prepare la requete d'update.
- **L41**: ligne vide.
- **L42**: execute la selection et recupere le resultat.
- **L43**: ligne vide.
- **L44**: boucle sur chaque ligne de vente non synchronisee.
- **L45**: ligne vide.
- **L46**: construit un objet `SaleRecord` depuis la ligne SQL.
- **L47**: met l'ID branche dans l'objet.
- **L48**: lit `sale_id`.
- **L49**: lit `product_name`.
- **L50**: lit `quantity`.
- **L51**: lit `unit_price`.
- **L52**: lit `sale_date` et convertit vers `LocalDateTime`.
- **L53**: fin construction objet.
- **L54**: ligne vide.
- **L55**: encode l'objet en message texte.
- **L56**: ligne vide.
- **L57**: publie le message vers RabbitMQ.
- **L58**: exchange cible.
- **L59**: routing key choisie au lancement.
- **L60**: message persistant (`deliveryMode=2`).
- **L61**: payload du message en UTF-8.
- **L62**: fin de publication.
- **L63**: ligne vide.
- **L64**: prepare le parametre `sale_id` pour update.
- **L65**: execute l'update `synced=true`.
- **L66**: ligne vide.
- **L67**: affiche la vente envoyee.
- **L68**: fin de boucle `while`.
- **L69**: ligne vide.
- **L70**: ferme le `ResultSet`.
- **L71**: ferme le `PreparedStatement` de selection.
- **L72**: ferme le `PreparedStatement` d'update.
- **L73**: ferme la connexion base BO.
- **L74**: ferme le channel RabbitMQ.
- **L75**: ferme la connexion RabbitMQ.
- **L76**: fin de `main`.
- **L77**: fin de la classe.


## 6) HeadOfficeConsumer.java


In [ ]:
import com.rabbitmq.client.*;

import java.nio.charset.StandardCharsets;
import java.sql.Connection;
import java.sql.PreparedStatement;
import java.sql.Timestamp;

public class HeadOfficeConsumer {

    public static void main(String[] args) throws Exception {

        ConnectionFactory factory = new ConnectionFactory();
        factory.setHost(AppConfig.RABBIT_HOST);

        com.rabbitmq.client.Connection mqConnection = factory.newConnection();
        Channel channel = mqConnection.createChannel();

        channel.exchangeDeclare(AppConfig.EXCHANGE_NAME, "direct", true);
        channel.queueDeclare(AppConfig.HO_QUEUE, true, false, false, null);

        channel.queueBind(AppConfig.HO_QUEUE, AppConfig.EXCHANGE_NAME, AppConfig.RK_BO1);
        channel.queueBind(AppConfig.HO_QUEUE, AppConfig.EXCHANGE_NAME, AppConfig.RK_BO2);

        channel.basicConsume(AppConfig.HO_QUEUE, false, (tag, delivery) -> {

            String msg = new String(delivery.getBody(), StandardCharsets.UTF_8);

            try {
                SaleRecord s = MessageCodec.decodeSaleRecord(msg);

                insertSaleRecord(s);

                channel.basicAck(delivery.getEnvelope().getDeliveryTag(), false);

                System.out.println("Recu: " + s);

            } catch (Exception e) {
                e.printStackTrace();
                channel.basicNack(delivery.getEnvelope().getDeliveryTag(), false, true);
            }

        }, tag -> {});
    }

    private static void insertSaleRecord(SaleRecord s) throws Exception {

        Connection conn = DbUtil.openConnection("sales_ho");

        String sql = "INSERT INTO consolidated_sales " +
                "(branch_id, sale_id, product_name, quantity, unit_price, sale_date) " +
                "VALUES (?, ?, ?, ?, ?, ?)";

        PreparedStatement ps = conn.prepareStatement(sql);

        ps.setString(1, s.branchId);
        ps.setInt(2, s.saleId);
        ps.setString(3, s.productName);
        ps.setInt(4, s.quantity);
        ps.setBigDecimal(5, s.unitPrice);
        ps.setTimestamp(6, Timestamp.valueOf(s.saleDate));

        ps.executeUpdate();

        ps.close();
        conn.close();
    }
}


## Explication ligne par ligne - HeadOfficeConsumer.java
- **L1**: importe toutes les classes client RabbitMQ.
- **L2**: ligne vide.
- **L3**: importe UTF-8 pour decoder le corps du message.
- **L4**: importe `Connection` JDBC.
- **L5**: importe `PreparedStatement` JDBC.
- **L6**: importe `Timestamp` pour l'insertion SQL.
- **L7**: ligne vide.
- **L8**: declare la classe consommateur du siege.
- **L9**: ligne vide.
- **L10**: point d'entree principal du consommateur.
- **L11**: ligne vide.
- **L12**: cree la factory RabbitMQ.
- **L13**: configure le host RabbitMQ.
- **L14**: ligne vide.
- **L15**: ouvre la connexion RabbitMQ.
- **L16**: ouvre le channel.
- **L17**: ligne vide.
- **L18**: declare l'exchange de synchro (durable).
- **L19**: declare la queue du siege.
- **L20**: ligne vide.
- **L21**: bind queue vers routing key BO1.
- **L22**: bind queue vers routing key BO2.
- **L23**: ligne vide.
- **L24**: lance la consommation manuelle (`autoAck=false`).
- **L25**: ligne vide.
- **L26**: convertit le corps binaire en String UTF-8.
- **L27**: ligne vide.
- **L28**: debut du `try` pour traiter le message.
- **L29**: decode le message en objet `SaleRecord`.
- **L30**: ligne vide.
- **L31**: insere la vente dans la base HO.
- **L32**: ligne vide.
- **L33**: envoie `ACK` au broker si tout est OK.
- **L34**: ligne vide.
- **L35**: affiche la vente recue.
- **L36**: ligne vide.
- **L37**: capture toute exception de traitement.
- **L38**: affiche la trace d'erreur.
- **L39**: envoie `NACK` avec requeue=true pour reessai.
- **L40**: fin du `catch`.
- **L41**: ligne vide.
- **L42**: fin de `basicConsume`; callback cancel vide.
- **L43**: fin de `main`.
- **L44**: ligne vide.
- **L45**: methode privee d'insertion dans HO.
- **L46**: ligne vide.
- **L47**: ouvre connexion a la base `sales_ho`.
- **L48**: ligne vide.
- **L49**: debut de requete SQL INSERT.
- **L50**: liste des colonnes cible de la table.
- **L51**: placeholders parametres `?` pour valeurs.
- **L52**: ligne vide.
- **L53**: prepare la requete SQL.
- **L54**: ligne vide.
- **L55**: affecte `branchId` au parametre 1.
- **L56**: affecte `saleId` au parametre 2.
- **L57**: affecte `productName` au parametre 3.
- **L58**: affecte `quantity` au parametre 4.
- **L59**: affecte `unitPrice` au parametre 5.
- **L60**: convertit `saleDate` vers `Timestamp` (parametre 6).
- **L61**: ligne vide.
- **L62**: execute l'insertion en base.
- **L63**: ligne vide.
- **L64**: ferme le statement.
- **L65**: ferme la connexion DB.
- **L66**: fin de la methode `insert`.
- **L67**: fin de la classe.


## Lancement du TP (section validation)

Utilise ces commandes dans un terminal (depuis `TP2`):

```powershell
# Compilation
javac -cp "lib/*;src" src/*.java -d bin

# Lancer le consommateur HO
java -cp "lib/*;bin" HeadOfficeConsumer

# Lancer BO1 (dans un autre terminal)
java -cp "lib/*;bin" BranchProducer BO1 sales_bo1 bo1.sales

# Lancer BO2 (dans un autre terminal)
java -cp "lib/*;bin" BranchProducer BO2 sales_bo2 bo2.sales

# Test style capture: insertion PreparedStatement
java -cp "lib/*;bin" JdbcPreparedTesting

# Test style capture: lecture ResultSet
java -cp "lib/*;bin" JdbcRetrieve
```

Points a verifier pendant la soutenance:
- Les lignes `Envoye: ...` apparaissent cote BO.
- Les lignes `Recu: ...` apparaissent cote HO.
- La table BO passe `synced` a `true`.
- La table HO (`consolidated_sales`) recoit les ventes des deux branches.
